In [15]:
import tensorflow as tf
import tensorflow_hub as hub
from tensorflow.keras import layers
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing import sequence
#!pip install tensorflow_hub
#importing all libraries

In [16]:
print("TF version:", tf.__version__)
print("TF Hub version:", hub.__version__)

# Create the embedding layer
hub_layer = hub.KerasLayer(
    "https://tfhub.dev/google/nnlm-en-dim128/2",
    input_shape=[],
    dtype=tf.string,
    trainable=False
)
# model = tf.keras.Model(inputs=inputs, outputs=outputs)

TF version: 2.18.0
TF Hub version: 0.16.1


In [20]:
hub_layer(tf.constant(["this is a test sentence"]))

<tf.Tensor: shape=(1, 128), dtype=float32, numpy=
array([[ 0.2625062 ,  0.12570179,  0.00881154, -0.05312935, -0.03127694,
        -0.02648829, -0.14021485,  0.04588968,  0.00646343,  0.09684554,
         0.1303938 , -0.25256464,  0.01854266, -0.02461666, -0.10922215,
        -0.12716603,  0.08009979, -0.08009237, -0.0110497 ,  0.27460146,
         0.01887552,  0.03947144, -0.04854007, -0.16459125,  0.17769106,
        -0.00068057, -0.01914907, -0.13387257, -0.02659447,  0.02184257,
        -0.05169556, -0.07221507,  0.04560696,  0.08399335,  0.06138731,
         0.10188591, -0.15495683, -0.03675179, -0.0896163 , -0.03958821,
        -0.05385005, -0.09905988, -0.04068294,  0.13841924, -0.02378968,
         0.04266504, -0.05567708, -0.06991865, -0.00390385,  0.03100423,
        -0.0329176 , -0.10160898, -0.2201724 ,  0.00497151,  0.03642236,
         0.09140317,  0.12536451,  0.01080319,  0.03432237, -0.12229932,
         0.01885111, -0.22963203, -0.06334151, -0.04905383,  0.00566361,
 

In [21]:
# Loading the data
max_features = 20000  # Only the top 20k words
maxlen = 200  # Only the first 200 words

In [22]:
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)
print(f"{len(x_train)} training sequences, {len(x_test)} test sequences")

25000 training sequences, 25000 test sequences


In [23]:
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)
# train the data

In [24]:
# Building the model
model = tf.keras.Sequential([
    layers.Embedding(max_features, 128, input_length=maxlen),
    layers.GlobalAveragePooling1D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.11/dist-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [25]:
# Compiling and training
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [26]:
history = model.fit(x_train, y_train,
                    batch_size=32,
                    epochs=5,
                    validation_data=(x_test, y_test))

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 28s 34ms/step - accuracy: 0.7072 - loss: 0.5449 - val_accuracy: 0.8686 - val_loss: 0.3053
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 42s 35ms/step - accuracy: 0.9069 - loss: 0.2353 - val_accuracy: 0.8768 - val_loss: 0.2995
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 26s 33ms/step - accuracy: 0.9364 - loss: 0.1703 - val_accuracy: 0.8647 - val_loss: 0.3388
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 41s 33ms/step - accuracy: 0.9546 - loss: 0.1347 - val_accuracy: 0.8434 - val_loss: 0.4478
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 41s 33ms/step - accuracy: 0.9650 - loss: 0.1048 - val_accuracy: 0.8523 - val_loss: 0.4380


In [27]:
model.summary() #summary of the model

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ (None, 200, 128)       │     2,560,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 7,729,925 (29.49 MB)

 Trainable params: 2,576,641 (9.83 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 5,153,284 (19.66 MB)

In [28]:
result = model.evaluate(x_test, y_test, verbose=2) # model evaluation

782/782 - 2s - 3ms/step - accuracy: 0.8523 - loss: 0.4380
